In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fma_catalog.gold.mlflow_tmp;

In [0]:
import mlflow
import mlflow.spark
import pickle
import os
import numpy as np
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import current_timestamp, col
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
catalog = dbutils.widgets.get("catalog_name")
silver_table = f"{catalog}.silver.audio_unsupervised"
gold_table = f"{catalog}.gold.audio_clusters"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.gold")

try:
    experiment_name = "/Shared/fma_unsupervised_clustering"
    mlflow.set_experiment(experiment_name)
    print(f"MLflow Experiment set to: {experiment_name}")
except Exception as e:
    # Fallback for restricted environments
    print("Shared path restricted, trying local workspace fallback...")
    mlflow.set_experiment("fma_clustering_local")

# Load the normalized features from Silver
silver_df = spark.table(silver_table)
feature_dim = silver_df.select("scaled_features").first()[0].size
print(f"Ready to train on {silver_df.count()} tracks with {feature_dim}-dimensional feature space.")
print(f"Note: Feature space is 74-dimensional (no PCA) or 20-30 dimensional (with PCA)")

In [0]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

evaluator = ClusteringEvaluator(
    predictionCol="prediction", 
    featuresCol="scaled_features", 
    metricName="silhouette", 
    distanceMeasure="squaredEuclidean"
)

print("Testing cluster sizes (K) with multiple metrics")
print("=" * 70)
print(f"{'K':<5} {'Silhouette':>12} {'Davies-Bouldin':>16} {'Calinski-Harabasz':>18} {'Training Cost':>15}")
print("=" * 70)

# Expanded K range for genre discovery (5-30)
for k in range(5, 31):
    with mlflow.start_run(run_name=f"KMeans_Search_K{k}", nested=True):
        # Train Model
        kmeans = KMeans(featuresCol="scaled_features", k=k, seed=42)
        model = kmeans.fit(silver_df)
        
        # Predictions
        predictions = model.transform(silver_df)
        
        # Metric 1: Silhouette Score (higher is better, -1 to 1)
        silhouette_score = evaluator.evaluate(predictions)
        
        # Metric 2: Davies-Bouldin Index (lower is better, 0+)
        # Measures average similarity ratio of each cluster with its most similar cluster
        predictions_pd = predictions.select("prediction", "scaled_features").toPandas()
        features_matrix = np.array([np.array(x) for x in predictions_pd["scaled_features"]])
        labels = predictions_pd["prediction"].values
        davies_bouldin = davies_bouldin_score(features_matrix, labels)
        
        # Metric 3: Calinski-Harabasz Index (higher is better)
        # Ratio of between-cluster dispersion to within-cluster dispersion
        calinski_harabasz = calinski_harabasz_score(features_matrix, labels)
        
        # Training cost (within-cluster sum of squares, for elbow method)
        cost = model.summary.trainingCost
        
        # Log to MLflow
        mlflow.log_param("k", k)
        mlflow.log_metric("silhouette_score", silhouette_score)
        mlflow.log_metric("davies_bouldin_index", davies_bouldin)
        mlflow.log_metric("calinski_harabasz_index", calinski_harabasz)
        mlflow.log_metric("training_cost", cost)
        
        print(f"{k:<5} {silhouette_score:>12.4f} {davies_bouldin:>16.4f} {calinski_harabasz:>18.2f} {cost:>15.2f}")

print("=" * 70)
print("\nMetric Interpretation:")
print("  - Silhouette: Higher is better (range -1 to 1, aim for > 0.3)")
print("  - Davies-Bouldin: Lower is better (aim for < 2.0)")
print("  - Calinski-Harabasz: Higher is better (no fixed range)")
print("  - Training Cost: Look for 'elbow' where cost reduction diminishes")
print("\nReview MLflow UI to select optimal K based on metric convergence.")

In [0]:
import mlflow.spark
import numpy as np

# IMPORTANT: Update OPTIMAL_K after reviewing metrics from the search above
# Expected range: 8-20 for music genres
# Choose K where:
#   - Silhouette score is highest (or near-highest)
#   - Davies-Bouldin is lowest
#   - Calinski-Harabasz is high
#   - Training cost shows "elbow" (diminishing returns)
OPTIMAL_K = 12  # UPDATE THIS after analyzing metrics in MLflow UI

mlflow_tmp_path = f"/Volumes/{catalog}/gold/mlflow_tmp"

with mlflow.start_run(run_name=f"Final_Gold_Model_K{OPTIMAL_K}"):
    print(f"Training final model with K={OPTIMAL_K}...")
    
    final_kmeans = KMeans(featuresCol="scaled_features", k=OPTIMAL_K, seed=42)
    final_model = final_kmeans.fit(silver_df)
    
    # Log Cluster Centers
    centers = final_model.clusterCenters()
    for i, center in enumerate(centers):
        mlflow.log_dict({"center_vector": center.tolist()}, f"cluster_centers/center_{i}.json")
    
    # Evaluate with all metrics
    predictions_df = final_model.transform(silver_df)
    final_silhouette = evaluator.evaluate(predictions_df)
    
    predictions_pd = predictions_df.select("prediction", "scaled_features").toPandas()
    features_matrix = np.array([np.array(x) for x in predictions_pd["scaled_features"]])
    labels = predictions_pd["prediction"].values
    final_davies_bouldin = davies_bouldin_score(features_matrix, labels)
    final_calinski_harabasz = calinski_harabasz_score(features_matrix, labels)
    
    # Log to MLflow Registry
    mlflow.log_param("optimal_k", OPTIMAL_K)
    mlflow.log_metric("final_silhouette", final_silhouette)
    mlflow.log_metric("final_davies_bouldin", final_davies_bouldin)
    mlflow.log_metric("final_calinski_harabasz", final_calinski_harabasz)
    
    mlflow.spark.log_model(
        spark_model=final_model, 
        artifact_path="spark_kmeans_model",
        dfs_tmpdir=mlflow_tmp_path 
    )
    
    native_save_path = f"{mlflow_tmp_path}/kmeans_model_v1"
    dbutils.fs.rm(native_save_path, True)
    final_model.save(native_save_path)
    
    print(f"Model registered and natively saved to Volume: {native_save_path}")
    print(f"Final Metrics: Silhouette={final_silhouette:.4f}, DB={final_davies_bouldin:.4f}, CH={final_calinski_harabasz:.2f}")

In [0]:
gold_df = (predictions_df
    .select("track_id", col("prediction").alias("cluster_id"))
    .withColumn("clustered_at", current_timestamp())
)

# Write to the Gold Delta table
(gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table))

print(f"Gold Layer Complete! {gold_df.count()} tracks successfully clustered.")